# IEX Simulation: Full Pipeline

**Steps:**
1. `IEX_STRUCTURED_V1` → apply Swap Sheet (swaps + self-changes) → `IEX_AFTER_ADJUSTMENT`
2. Both sources → 30-min intervals → `IEX_INTERVALS_ORIGINAL` / `IEX_INTERVALS_AFTER_ADJUSTMENT`
3. Excel (5 sheets) + HTML interval comparison
4. `IEX_ACTUAL_ADJUSTMENT` (raw IEX post-swap) → parse → side-by-side audit vs `IEX_AFTER_ADJUSTMENT`

In [1]:
import pandas as pd
import numpy as np
import os, pathlib, time, shutil, tempfile
from datetime import datetime as _dt, timedelta
from IPython.display import display, HTML
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
try:
    import polars as pl; HAS_POLARS = True
except ImportError:
    HAS_POLARS = False

In [2]:
WEEK_MONDAY = '2026-09-07'   # ← Change this to switch weeks

_week    = pd.Timestamp(WEEK_MONDAY)
_week_end = _week + pd.Timedelta(days=6)
print(f"Week : {_week.strftime('%A %d %b %Y')} → {_week_end.strftime('%A %d %b %Y')}")

Week : Monday 07 Sep 2026 → Sunday 13 Sep 2026


In [3]:
_home = os.path.expanduser('~').replace('\\', '/')
_base = f'{_home}/Concentrix Corporation/WFM-Expedia-HCM - Branding files'
_sim  = f'{_base}/Rawdata/INPUT_SCHEDULE/SCHEDULE_SIMULATION'

PATHS = {
    'structured_v1'              : f'{_sim}/IEX_STRUCTURED_V1',
    'after_adjustment'           : f'{_sim}/IEX_AFTER_ADJUSTMENT',
    'actual_adjustment'          : f'{_sim}/IEX_ACTUAL_ADJUSTMENT',
    'intervals_original'         : f'{_sim}/IEX_INTERVALS_ORIGINAL',
    'intervals_after_adjustment' : f'{_sim}/IEX_INTERVALS_AFTER_ADJUSTMENT',
    'rta_folder'                 : f'{_base}/BI_Task/Schedule (RTA version)/2026',
    'excel_output'               : _sim,
    'ou_mail_folder'             : f'{_base}/Rawdata/INPUT_OU_MAIL',
    'intervals_actual'           : f'{_sim}/IEX_INTERVALS_ACTUAL',
}
for k, v in PATHS.items():
    os.makedirs(v, exist_ok=True)
    print(f"  {'OK' if os.path.exists(v) else 'MISSING':8s}  {k}")

ALL_VNT_INTERVALS = [
    f"{h:02d}:{m:02d}-{(h+(m+30)//60)%24:02d}:{(m+30)%60:02d}"
    for h in range(24) for m in (0, 30)
]
WEEKDAYS = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

  OK        structured_v1
  OK        after_adjustment
  OK        actual_adjustment
  OK        intervals_original
  OK        intervals_after_adjustment
  OK        rta_folder
  OK        excel_output
  OK        ou_mail_folder
  OK        intervals_actual


In [4]:
# ── STEP 0: Parse IEX_ORIGINAL_V1 → IEX_STRUCTURED_V1 ───────────────────────

ws_str   = _week.strftime('%Y_%m_%d')
out_file = pathlib.Path(PATHS['structured_v1']) / f'IEX_Structured_{ws_str}.csv'

if out_file.exists():
    print(f'IEX_Structured_{ws_str}.csv already exists — skipping parse.')
else:
    print(f'Parsing IEX_ORIGINAL_V1 → IEX_Structured_{ws_str}.csv ...')

    RENAME_MAP = {
        'Agent Schedules':'Agent','__UNNAMED__1':'Date','__UNNAMED__2':'Start_Shift',
        '__UNNAMED__3':'End_Shift','__UNNAMED__5':'Scheduled Activity',
        '__UNNAMED__6':'Start_Action','__UNNAMED__9':'End_Action',
    }

    list_dfs = []
    orig_folder = pathlib.Path(PATHS['structured_v1']).parent / 'IEX_ORIGINAL_V1'
    if not orig_folder.exists():
        orig_folder = pathlib.Path(
            f'{os.path.expanduser("~").replace(chr(92),"/")}'
            '/Concentrix Corporation/WFM-Expedia-HCM - Branding files'
            '/Rawdata/INPUT_SCHEDULE/SCHEDULE_SIMULATION/IEX_ORIGINAL_V1')
    print(f'  Source folder: {orig_folder}')

    for fp in list(orig_folder.glob('**/*.xlsx')) + list(orig_folder.glob('**/*.csv')):
        if fp.name.startswith('~$'): continue
        export_dt = _dt(*time.localtime(os.path.getmtime(fp))[:6])
        try:
            if HAS_POLARS:
                raw = pl.read_excel(fp, infer_schema_length=0) if fp.suffix.lower()=='.xlsx' \
                      else pl.read_csv(fp, infer_schema_length=0, encoding='utf-8', ignore_errors=True)
                df  = raw.to_pandas()
            else:
                df = pd.read_excel(fp, dtype=str) if fp.suffix.lower()=='.xlsx' \
                     else pd.read_csv(fp, dtype=str, encoding='utf-8', errors='ignore')
            df['sheet_name']  = fp.stem
            df['Export time'] = export_dt
            list_dfs.append(df)
            print(f'  Read: {fp.name} ({len(df)} rows)')
        except Exception as ex:
            print(f'  Error: {fp.name} — {ex}')

    if not list_dfs:
        print(f'  No files found in {orig_folder}')
        print('  Place raw IEX xlsx files in IEX_ORIGINAL_V1 and re-run.')
    else:
        IEX = pd.concat(list_dfs, ignore_index=True)
        IEX = IEX.rename(columns={k:v for k,v in RENAME_MAP.items() if k in IEX.columns})

        if 'Agent' not in IEX.columns:
            print(f'  Column "Agent" not found. Columns: {list(IEX.columns[:10])}')
        else:
            IEX['Generate Date'] = np.where(
                IEX['Agent'].str.contains('Generation Date: ', na=False),
                IEX['Agent'].str.extract(r'Generation Date: (.+)')[0], np.nan)
            IEX['Generate Date'] = IEX['Generate Date'].bfill()
            IEX['Agent'] = IEX['Agent'].ffill()

            off_table = IEX[IEX.get('Start_Shift', pd.Series(dtype=str)) == 'Off'].copy()
            iex_edit  = IEX[
                (IEX.get('Start_Shift', pd.Series(dtype=str)) != 'Off') &
                (IEX.get('Date',        pd.Series(dtype=str)) != 'Date') &
                ~(IEX.get('Date', pd.Series()).isna() & IEX.get('Scheduled Activity', pd.Series()).isna())
            ].copy()

            for col in ['Date','Start_Shift','End_Shift']:
                if col in iex_edit.columns: iex_edit[col] = iex_edit[col].ffill()

            iex_edit = iex_edit[
                iex_edit['Agent'].str.contains('Agent: ', na=False) | iex_edit['Agent'].isna()]
            if 'Scheduled Activity' in iex_edit.columns:
                iex_edit = iex_edit[iex_edit['Scheduled Activity'].notna()]

            if 'Start_Shift' in off_table.columns and 'Scheduled Activity' in off_table.columns:
                off_table['Scheduled Activity'] = off_table['Scheduled Activity'].fillna(off_table['Start_Shift'])
                off_table['Start_Shift'] = off_table['Start_Shift'].replace('Off', np.nan)

            result = pd.concat([off_table, iex_edit], axis=0, ignore_index=True)
            result['Date'] = pd.to_datetime(result.get('Date'), errors='coerce')

            result = result[(result['Date'] >= _week) & (result['Date'] <= _week_end)]
            result = result.sort_values(['Agent','Date','Start_Action'], na_position='first')

            drop_unnamed = [c for c in result.columns if str(c).startswith('__UNNAMED__')]
            result = result.drop(columns=drop_unnamed)
            result.to_csv(out_file, index=False, encoding='utf-8-sig')
            print(f'  ✅ Saved: {out_file.name} ({len(result):,} rows)')


IEX_Structured_2026_09_07.csv already exists — skipping parse.


In [5]:

def safe_read_excel(src, **kw):
    tmp = os.path.join(tempfile.gettempdir(), f'_tmp_{os.path.basename(src)}')
    shutil.copy2(src, tmp)
    try:    return pd.read_excel(tmp, **kw)
    finally:
        try: os.remove(tmp)
        except: pass

def parse_ampm_to_minutes(val):
    if pd.isna(val) or str(val).strip().lower() in ('','nan','none'): return None
    s = str(val).strip()
    for fmt in ('%I:%M %p','%H:%M:%S','%H:%M'):
        try:
            t = _dt.strptime(s, fmt)
            return t.hour*60 + t.minute
        except: continue
    return None

def minutes_to_ampm(minutes):
    m = int(minutes) % 1440
    h, mn = divmod(m, 60)
    return f"{h%12 or 12}:{mn:02d} {'AM' if h<12 else 'PM'}"

def parse_shift_code(shift_str):
    if not isinstance(shift_str, str) or '-' not in shift_str: return None, None
    parts = shift_str.strip().split('-')
    if len(parts) != 2: return None, None
    try:
        s = parts[0].strip().zfill(4); e = parts[1].strip().zfill(4)
        return int(s[:2])*60+int(s[2:4]), int(e[:2])*60+int(e[2:4])
    except: return None, None

def shift_rows_by_offset(rows, offset_mins):
    result = rows.copy()
    for col in ['Start_Shift','End_Shift','Start_Action','End_Action']:
        if col not in result.columns: continue
        result[col] = result[col].apply(
            lambda v: minutes_to_ampm(parse_ampm_to_minutes(v)+offset_mins)
                      if parse_ampm_to_minutes(v) is not None else v)
    return result

def get_shift_start_mins(rows):
    for _, r in rows.iterrows():
        m = parse_ampm_to_minutes(r.get('Start_Shift'))
        if m is not None: return m
    return None

def generate_default_schedule(agent_template, date_val, new_shift_str):
    s_mins, e_mins = parse_shift_code(new_shift_str)
    if s_mins is None: return pd.DataFrame()
    dur = e_mins-s_mins if e_mins>s_mins else e_mins+1440-s_mins
    lunch_s = s_mins+dur//2-30; lunch_e = lunch_s+60
    b1_s = (s_mins+lunch_s)//2; b1_e = b1_s+15
    end_abs = s_mins+dur; b2_s = (lunch_e+end_abs)//2; b2_e = b2_s+15
    acts = [('Open Time',s_mins,b1_s),('Break',b1_s,b1_e),('Open Time',b1_e,lunch_s),
            ('Lunch',lunch_s,lunch_e),('Open Time',lunch_e,b2_s),('Break',b2_s,b2_e),('Open Time',b2_e,end_abs)]
    ss = minutes_to_ampm(s_mins); es = minutes_to_ampm(e_mins)
    rows = []
    for act,ts,te in acts:
        row = dict(agent_template)
        row.update({'Date':date_val,'Start_Shift':ss,'End_Shift':es,'Scheduled Activity':act,
                    'Start_Action':minutes_to_ampm(ts),'End_Action':minutes_to_ampm(te)})
        rows.append(row)
    return pd.DataFrame(rows)

IDENTITY_COLS  = ['Agent','Generate Date','Export time','sheet_name']
SKIP_VALUES    = {'KEEP','SWAP','NAN','NONE',''}
SINGLE_ROW_MAP = {'OFF':'Off','TERMINATION':'Termination','HO':'Holiday','AL':'AL',
                  'CO':'CO','LWP':'LWP','PTO':'PTO','NCNS':'No Call/No Show'}

def swap_agent_day(df, iex1, iex2, date_val):
    m1=(df['IEX_ID']==iex1)&(df['Date']==date_val)
    m2=(df['IEX_ID']==iex2)&(df['Date']==date_val)
    r1,r2=df[m1].copy(),df[m2].copy()
    if r1.empty or r2.empty:
        missing=[f'IEX {x}' for x,r in[(iex1,r1),(iex2,r2)] if r.empty]
        print(f'    [WARN] {date_val.date()}: {", ".join(missing)} no rows → skip')
        return df
    id1={c:r1.iloc[0][c] for c in IDENTITY_COLS if c in r1.columns}
    id2={c:r2.iloc[0][c] for c in IDENTITY_COLS if c in r2.columns}
    n1=r2.copy(); [n1.__setitem__(c,v) for c,v in id1.items()]; n1['IEX_ID']=iex1
    n2=r1.copy(); [n2.__setitem__(c,v) for c,v in id2.items()]; n2['IEX_ID']=iex2
    s1=r1['Start_Shift'].iloc[0] if 'Start_Shift' in r1.columns else '?'
    s2=r2['Start_Shift'].iloc[0] if 'Start_Shift' in r2.columns else '?'
    print(f'    [SWAP] {date_val.date()}: {iex1}({s1}) ↔ {iex2}({s2}) | rows:{len(r1)}↔{len(r2)}')
    return pd.concat([df[~(m1|m2)],n1,n2],ignore_index=True)

def apply_self_change(df, iex_id, date_val, new_value):
    mask=(df['IEX_ID']==iex_id)&(df['Date']==date_val)
    existing=df[mask].copy(); val_up=str(new_value).strip().upper()
    if val_up in SINGLE_ROW_MAP:
        act_label=SINGLE_ROW_MAP[val_up]
        agent_all=df[df['IEX_ID']==iex_id]
        tpl=existing.iloc[0].copy() if not existing.empty else (agent_all.iloc[0].copy() if not agent_all.empty else pd.Series())
        if tpl.empty: return df
        new_row=tpl.copy()
        new_row['Date']=date_val; new_row['Scheduled Activity']=act_label
        for c in ['Start_Shift','End_Shift','Start_Action','End_Action']: new_row[c]=np.nan
        old=existing['Start_Shift'].iloc[0] if not existing.empty else 'OFF'
        print(f'    [MOVE] IEX {iex_id} {date_val.date()}: {old} → {act_label}')
        return pd.concat([df[~mask],pd.DataFrame([new_row])],ignore_index=True)
    new_start,new_end=parse_shift_code(new_value)
    if new_start is None:
        print(f'    [WARN] IEX {iex_id}: cannot parse "{new_value}" → skip'); return df
    ns=minutes_to_ampm(new_start); ne=minutes_to_ampm(new_end)
    if not existing.empty:
        old_start=get_shift_start_mins(existing)
        if old_start is None: return df
        new_rows=shift_rows_by_offset(existing,new_start-old_start)
        new_rows['Start_Shift']=ns; new_rows['End_Shift']=ne
        df=pd.concat([df[~mask],new_rows],ignore_index=True)
        print(f'    [MOVE] IEX {iex_id} {date_val.date()}: offset {new_start-old_start:+d}min → {ns}')
        return df
    agent_all=df[df['IEX_ID']==iex_id].copy()
    prev_rows=pd.DataFrame()
    if not agent_all.empty:
        prev_dates=agent_all[agent_all['Date']<date_val]['Date'].dropna().unique()
        if len(prev_dates):
            prev_day=max(prev_dates); cand=agent_all[agent_all['Date']==prev_day].copy()
            if get_shift_start_mins(cand) is not None: prev_rows=cand
    if not prev_rows.empty:
        old_start=get_shift_start_mins(prev_rows)
        new_rows=shift_rows_by_offset(prev_rows,new_start-old_start)
        new_rows['Date']=date_val; new_rows['Start_Shift']=ns; new_rows['End_Shift']=ne
        src_info=f'from {prev_rows.iloc[0]["Date"].date()}, offset {new_start-old_start:+d}min'
    else:
        tpl=agent_all.iloc[0].to_dict() if not agent_all.empty else {}
        new_rows=generate_default_schedule(tpl,date_val,new_value)
        src_info='default 9h schedule'
    if new_rows.empty: return df
    df=pd.concat([df[~mask],new_rows],ignore_index=True)
    print(f'    [MOVE] IEX {iex_id} {date_val.date()}: OFF → {ns} ({src_info})')
    return df

def week_date_to_rta_path(monday, rta_folder):
    return os.path.join(rta_folder, f"Schedule_WB{monday.strftime('%m%d')}.xlsx")

def load_swap_sheet(rta_path, week_str):
    if not os.path.exists(rta_path):
        print(f'  [WARN] RTA not found: {rta_path}'); return pd.DataFrame()
    try:
        raw=safe_read_excel(rta_path,sheet_name='Swap',header=None)
        raw.columns=raw.iloc[0]; df=raw.iloc[1:].reset_index(drop=True)
    except Exception as e:
        print(f'  [ERR] Swap sheet: {e}'); return pd.DataFrame()
    if 'Week' in df.columns:
        df=df[df['Week'].astype(str).str.strip()==week_str].copy()
    for col in ['IEX1','IEX2']:
        if col in df.columns:
            df[col]=pd.to_numeric(df[col],errors='coerce').astype('Int64')
    SCOLS=['Week','IEX1','IEX2','Name1','Name2','Step',
           'Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    df=df[[c for c in SCOLS if c in df.columns]].reset_index(drop=True)
    df['_row_order_']=df.index
    print(f'  Swap rows for {week_str}: {len(df)}')
    return df

def extract_iex_id(agent_series):
    return agent_series.str.extract(r'(\d+)',expand=False).astype('Int64')

def load_planned_sheet(rta_path, week_str):
    if not os.path.exists(rta_path):
        return pd.DataFrame()
    try:
        raw = safe_read_excel(rta_path, sheet_name='Planned', header=None)
        hrow = 0
        for ridx, row in raw.iterrows():
            if any('IEX ID' in str(v) or 'Leave Date' in str(v) for v in row.values):
                hrow = ridx; break
        raw.columns = raw.iloc[hrow].astype(str)
        df = raw.iloc[hrow+1:].reset_index(drop=True)
        df = df.rename(columns=lambda c: c.strip())
        df = df.dropna(how='all')
        id_col   = next((c for c in df.columns if 'IEX' in c.upper() and 'ID' in c.upper()), None)
        date_col = next((c for c in df.columns if 'LEAVE' in c.upper() and 'DATE' in c.upper()), None)
        type_col = next((c for c in df.columns if 'TYPE' in c.upper()), None)
        if not all([id_col, date_col, type_col]):
            print(f'  [WARN] Planned sheet columns not found: id={id_col} date={date_col} type={type_col}')
            return pd.DataFrame()
        df = df[[id_col, date_col, type_col]].copy()
        df.columns = ['IEX_ID','Leave_Date','Type']
        df['IEX_ID']     = pd.to_numeric(df['IEX_ID'], errors='coerce').astype('Int64')
        df['Leave_Date'] = pd.to_datetime(df['Leave_Date'], errors='coerce')
        df['Type']       = df['Type'].astype(str).str.strip().str.upper()
        df = df.dropna(subset=['IEX_ID','Leave_Date'])
        print(f'  Planned sheet: {len(df)} entries | Types: {df["Type"].value_counts().to_dict()}')
        return df
    except Exception as e:
        print(f'  [WARN] Cannot load Planned sheet: {e}')
        return pd.DataFrame()

IEX_ACT_TO_PLANNED_TYPE = {
    'ptoal':        'AL',
    'paid leave':   'CO',
    'unpaid leave': 'LWP',
    'al':           'AL',
    'co':           'CO',
    'lwp':          'LWP',
}

print("Swap pipeline helpers loaded.")

Swap pipeline helpers loaded.


In [6]:
week_str = f"Schedule_{_week.strftime('%Y_%m_%d')}"
rta_path = week_date_to_rta_path(_week, PATHS['rta_folder'])

print(f"Week string : {week_str}")
print(f"RTA file    : {os.path.basename(rta_path)} {'✅' if os.path.exists(rta_path) else '❌ NOT FOUND'}")

ws_str = _week.strftime('%Y_%m_%d')
src_file = next((pathlib.Path(PATHS['structured_v1'])/n
                 for n in [f'IEX_Structured_{ws_str}.csv']
                 if (pathlib.Path(PATHS['structured_v1'])/n).exists()), None)

if src_file is None:
    print(f"[WARN] IEX_Structured_{ws_str}.csv not found in IEX_STRUCTURED_V1")
    print("       Run IEX_parse_to_structured.ipynb first to generate it.")
    iex_df = pd.DataFrame()
else:
    iex_df = pd.read_csv(src_file, dtype=str)
    iex_df['Date']   = pd.to_datetime(iex_df['Date'], errors='coerce')
    iex_df['IEX_ID'] = extract_iex_id(iex_df['Agent'])
    print(f"Loaded IEX_STRUCTURED_V1: {len(iex_df):,} rows | {iex_df['IEX_ID'].nunique()} agents")

swap_df = load_swap_sheet(rta_path, week_str)
day_to_date = {day: (_week+timedelta(days=i)).normalize() for i,day in enumerate(WEEKDAYS)}

if not iex_df.empty and not swap_df.empty:
    swap_count = 0; self_count = 0

    for _, row in swap_df.sort_values('_row_order_').iterrows():
        iex1=row.get('IEX1'); iex2=row.get('IEX2'); step=str(row.get('Step','?')).strip()
        name1=str(row.get('Name1','')).strip(); name2=str(row.get('Name2','')).strip()
        is_self=pd.isna(iex2) or (pd.notna(iex1) and pd.notna(iex2) and iex1==iex2)
        if pd.isna(iex1): continue
        iex1_int=int(iex1)

        if is_self:
            print(f'\n  ── Step {step} [SELF]: {iex1_int} ({name1})')
            for day in WEEKDAYS:
                cell_val=str(row.get(day,'')).strip()
                if not cell_val or cell_val.upper() in SKIP_VALUES: continue
                date_val=day_to_date.get(day)
                if date_val is None: continue
                iex_df=apply_self_change(iex_df,iex1_int,date_val,cell_val); self_count+=1
        else:
            if pd.isna(iex2): continue
            iex2_int=int(iex2)
            print(f'\n  ── Step {step} [SWAP]: {iex1_int}({name1}) ↔ {iex2_int}({name2})')
            for day in WEEKDAYS:
                flag=str(row.get(day,'')).strip().upper()
                if flag!='SWAP': continue
                date_val=day_to_date.get(day)
                if date_val is None: continue
                has1=((iex_df['IEX_ID']==iex1_int)&(iex_df['Date']==date_val)).any()
                has2=((iex_df['IEX_ID']==iex2_int)&(iex_df['Date']==date_val)).any()
                if has1 and has2:
                    iex_df=swap_agent_day(iex_df,iex1_int,iex2_int,date_val); swap_count+=1
                else:
                    missing=[f'IEX {x}' for x,h in[(iex1_int,has1),(iex2_int,has2)] if not h]
                    print(f'    [WARN] {day}: {", ".join(missing)} no rows → skip (WO/Off?)')

    iex_df=iex_df.sort_values(['Agent','Date','Start_Action'],na_position='first').reset_index(drop=True)
    out_name=f'IEX_Adjusted_{ws_str}.csv'
    out_path=os.path.join(PATHS['after_adjustment'],out_name)
    iex_df.drop(columns=['IEX_ID'],errors='ignore').to_csv(out_path,index=False,encoding='utf-8-sig')
    print(f'\n✅ {out_name} saved ({len(iex_df):,} rows | {swap_count} swaps | {self_count} self-changes)')

elif iex_df.empty:
    print("[INFO] No IEX structured data — skipping swap. Run IEX_parse_to_structured first.")
else:
    print("[INFO] No swap sheet data — copying IEX_STRUCTURED_V1 as-is to IEX_AFTER_ADJUSTMENT.")
    out_path=os.path.join(PATHS['after_adjustment'],f'IEX_Adjusted_{ws_str}.csv')
    iex_df.drop(columns=['IEX_ID'],errors='ignore').to_csv(out_path,index=False,encoding='utf-8-sig')
    print(f"  Saved: {os.path.basename(out_path)}")

Week string : Schedule_2026_09_07
RTA file    : Schedule_WB0907.xlsx ✅
Loaded IEX_STRUCTURED_V1: 4,168 rows | 117 agents
  Swap rows for Schedule_2026_09_07: 42

  ── Step 1 [SWAP]: 3052052(NGUYEN THANH LUAN) ↔ 3109357(VO DUC HUY)
    [SWAP] 2026-09-07: 3052052(nan) ↔ 3109357(9:00 AM) | rows:1↔7
    [SWAP] 2026-09-08: 3052052(nan) ↔ 3109357(9:00 AM) | rows:1↔7
    [SWAP] 2026-09-09: 3052052(9:00 AM) ↔ 3109357(9:00 AM) | rows:7↔7
    [SWAP] 2026-09-10: 3052052(9:00 AM) ↔ 3109357(nan) | rows:7↔1
    [SWAP] 2026-09-11: 3052052(9:00 AM) ↔ 3109357(nan) | rows:7↔1
    [SWAP] 2026-09-12: 3052052(9:00 AM) ↔ 3109357(9:00 AM) | rows:7↔7
    [SWAP] 2026-09-13: 3052052(9:00 AM) ↔ 3109357(9:00 AM) | rows:7↔7

  ── Step 1 [SELF]: 3052052 (NGUYEN THANH LUAN)
    [MOVE] IEX 3052052 2026-09-07: 9:00 AM → Off
    [MOVE] IEX 3052052 2026-09-08: 9:00 AM → Off
    [MOVE] IEX 3052052 2026-09-09: offset +720min → 9:00 PM
    [MOVE] IEX 3052052 2026-09-12: offset +720min → 9:00 PM
    [MOVE] IEX 3052052 2026-

In [7]:
SCHEDULED_ACTIVITIES = {
    'Open Time','Extra Hours','No Call/No Show','PTO','Training Offline',
    'Sick Leave','Paid Leave','Termination','Off Phone Misc',
    'Billable Training','Nesting Training','Training',
}

def convert_to_time(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], format='%I:%M %p', errors='coerce').dt.time
    return df

def create_datetime_vec(df):
    base = pd.to_datetime(df['Date'])
    def _c(b, t): return pd.to_datetime(b.astype(str)+' '+t.astype(str), errors='coerce')
    s=_c(base,df['Start_Action']); e=_c(base,df['End_Action']); sh=_c(base,df['Start_Shift'])
    s+=pd.to_timedelta(((df['Scheduled']!=0)&s.notna()&sh.notna()&(s<sh)).astype(int),unit='D')
    e+=pd.to_timedelta(((df['Scheduled']!=0)&e.notna()&s.notna()&(e<s)).astype(int),unit='D')
    return s.where(df['Scheduled']!=0,pd.NaT), e.where(df['Scheduled']!=0,pd.NaT)

def _shift_range(df, mask, prefix):
    agg=(df[mask].groupby(['Date','IEX_ID'],sort=False)
         .agg(**{f'Datetime_{prefix}_Start_Shift':('Datetime_Start_Action','min'),
                f'Datetime_{prefix}_End_Shift'  :('Datetime_End_Action','max')})
         .reset_index())
    sc,ec=f'Datetime_{prefix}_Start_Shift',f'Datetime_{prefix}_End_Shift'
    agg[f'{prefix} Shift']=agg[sc].dt.strftime('%H%M')+'-'+agg[ec].dt.strftime('%H%M')
    return agg

def build_sorted_df(csv_path):
    df=pd.read_csv(csv_path,dtype=str)
    df['Date']=pd.to_datetime(df['Date'],errors='coerce')
    df['IEX_ID']=df['Agent'].str.extract(r'(\d+)',expand=False).astype('Int64')
    df['Month']=df['Date'].dt.strftime('%b-%y')
    df['Week_Monday']=(df['Date']-pd.to_timedelta(df['Date'].dt.dayofweek,unit='D')).dt.normalize()
    df['Agent Name']=df['Agent'].str.extract(r'\d+ (.+)',expand=False).str.upper()
    df['Scheduled']=df.groupby(['Date','IEX_ID'])['Scheduled Activity'].transform(
        lambda x: 1 if x.isin(SCHEDULED_ACTIVITIES).any() else 0)
    if 'Generate Date' in df.columns:
        df['Generate Date']=pd.to_datetime(df['Generate Date'],errors='coerce')
        mg=df.groupby(['IEX_ID','Date'],sort=False)['Generate Date'].max().reset_index(name='_mg')
        df=df.merge(mg,on=['IEX_ID','Date'],how='left')
        df=df[df['Generate Date']==df['_mg']].drop(columns=['_mg'])
    df=convert_to_time(df,['Start_Shift','End_Shift','Start_Action','End_Action'])
    df['Datetime_Start_Action'],df['Datetime_End_Action']=create_datetime_vec(df)
    df['_tr']=df['Scheduled Activity'].str.contains('training',case=False,na=False)
    otk=set(df.loc[df['Scheduled Activity']=='Open Time',['Date','IEX_ID']].itertuples(index=False,name=None))
    df['_ho']=[(d,i) in otk for d,i in zip(df['Date'],df['IEX_ID'])]
    fl=df['Scheduled Activity'].isin(['Open Time','Extra Hours','No Call/No Show','System Outage','Offline'])|df['_tr']
    fi=(df['_ho']&(df['Scheduled Activity'].isin(['Open Time','No Call/No Show','System Outage','Offline'])|df['_tr']))|~df['_ho']
    df=(df.merge(_shift_range(df,fl,'Fluctuate'),on=['Date','IEX_ID'],how='left')
          .merge(_shift_range(df,fi,'First'),on=['Date','IEX_ID'],how='left'))
    df=df[~(df['Scheduled Activity'].isin(['Lunch','Break'])&df['Fluctuate Shift'].isna())].copy()
    df['Duration']=(df['Datetime_End_Action']-df['Datetime_Start_Action']).dt.total_seconds()
    df['Time_Of_Day']=(df['Datetime_First_End_Shift']-df['Datetime_First_Start_Shift']).dt.total_seconds()/3600
    df=df.sort_values(['Date','IEX_ID','Datetime_Start_Action'],na_position='last').drop_duplicates()
    TRACT=['Training Offline','Training','Training Nesting','Nesting Training','Billable Training']
    AMAP={'Open Time':'Open Time','Break':'Break Time','Lunch':'Lunch Time','Extra Hours':'Extra Time','No Call/No Show':'NCNS','PTO':'AL'}
    tr=(df[df['Scheduled Activity'].isin(TRACT)].groupby(['Date','IEX_ID'],sort=False)['Duration'].sum().div(3600).reset_index(name='Training'))
    df=df.merge(tr,on=['Date','IEX_ID'],how='left')
    tt=(df[df['Scheduled Activity'].isin(AMAP)].groupby(['Date','IEX_ID','Scheduled Activity'],sort=False)['Duration']
        .sum().div(3600).reset_index().assign(col_name=lambda d:d['Scheduled Activity'].map(AMAP))
        .pivot_table(index=['Date','IEX_ID'],columns='col_name',values='Duration',aggfunc='sum').reset_index())
    tt.columns.name=None; df=df.merge(tt,on=['Date','IEX_ID'],how='left')
    df=df.sort_values(['Date','IEX_ID','Datetime_Start_Action'],na_position='last')
    df['FSA']=df.groupby(['Date','IEX_ID'])['Scheduled Activity'].transform('first')
    fsa=df['FSA']; ot=df.get('Open Time',pd.Series(np.nan,index=df.index))
    et=df.get('Extra Time',pd.Series(np.nan,index=df.index)); nc=df.get('NCNS',pd.Series(0,index=df.index))
    conds=[fsa.isin(['Holiday','Bereavement','Off','Off Phone Misc','Unscheduled']),fsa=='PTO',
           fsa.isin(['Sickness','Sick Leave']),fsa.isin(['Training Offline','Billable Training','Nesting Training']),
           fsa.isin(['Paid Leave']),fsa.isin(['Leave']),fsa.isin(['Termination']),
           (fsa=='Extra Hours')&ot.isna(),(ot>0)&(nc>0)&(nc<=5),((ot==0)|ot.isna())&(nc>5),
           (ot>5)&(et>0),(ot>0)&(ot<5)&(et>0),(ot==0)&(et>0),(ot>0)]
    choices=[fsa,'AL','SL','Training Offline','CO','LWP','Termination','PO','HDL','NCNS','PR - OT','HDL - OT','PO','PR']
    df['Shift Tracking']=np.select(conds,choices,default=fsa)
    df['First Shift']=np.where(df['Time_Of_Day'].isna(),df['Scheduled Activity'],df['First Shift'])
    return df.drop(columns=['_tr','_ho','FSA'],errors='ignore')

def generate_intervals(df_input):
    df=df_input.copy()
    df['Datetime_Start_Action']=pd.to_datetime(df['Datetime_Start_Action'],errors='coerce')
    df['Datetime_End_Action']  =pd.to_datetime(df['Datetime_End_Action'],  errors='coerce')
    df=df.dropna(subset=['Datetime_Start_Action','Datetime_End_Action'])
    ov=df['Datetime_End_Action']<df['Datetime_Start_Action']
    df.loc[ov,'Datetime_End_Action']+=pd.Timedelta(days=1)
    df['_fl']=df['Datetime_Start_Action'].dt.floor('30min')
    df['_ce']=df['Datetime_End_Action'].dt.ceil('30min')
    df['_n']=(((df['_ce']-df['_fl'])/pd.Timedelta(minutes=30)).fillna(0).astype(int))
    df=df[df['_n']>0].copy()
    e=df.loc[df.index.repeat(df['_n'])].copy()
    e['_i']=e.groupby(level=0).cumcount()
    e['Datetime_Start_Time_Full']=e['_fl']+pd.to_timedelta(e['_i']*30,unit='m')
    e['Datetime_End_Time_Full']  =e['Datetime_Start_Time_Full']+pd.Timedelta(minutes=30)
    e['Datetime_Start_Time']=np.maximum(e['Datetime_Start_Time_Full'],e['Datetime_Start_Action'])
    e['Datetime_End_Time']  =np.minimum(e['Datetime_End_Time_Full'],  e['Datetime_End_Action'])
    e=e[e['Datetime_Start_Time']<e['Datetime_End_Time']].copy()
    e['Duration']=(e['Datetime_End_Time']-e['Datetime_Start_Time']).dt.total_seconds()/3600
    e['VNT_Intervals']=e['Datetime_Start_Time_Full']
    e['PST_Intervals']=(e['VNT_Intervals'].dt.tz_localize('Asia/Ho_Chi_Minh').dt.tz_convert('America/Los_Angeles').dt.tz_localize(None))
    vs=e['VNT_Intervals'].dt.strftime('%H:%M'); ve=(e['VNT_Intervals']+pd.Timedelta(minutes=30)).dt.strftime('%H:%M')
    ps=e['PST_Intervals'].dt.strftime('%H:%M'); pe=(e['PST_Intervals']+pd.Timedelta(minutes=30)).dt.strftime('%H:%M')
    e['VNT_Interval_Range']=vs+'-'+ve; e['PST_Interval_Range']=ps+'-'+pe
    e['Work Category']=np.where(e['Scheduled Activity'].isin(['Open Time','Extra Hours']),'Productive','Unproductive')
    e=e.rename(columns={'Date':'Date_Converted','IEX_ID':'IEX ID'})
    COLS=['Month','Week_Monday','Date_Converted','Agent Name','IEX ID','First Shift',
          'Scheduled Activity','VNT_Intervals','PST_Intervals','VNT_Interval_Range','PST_Interval_Range',
          'Datetime_Start_Time','Datetime_End_Time','Duration','Work Category']
    return e[[c for c in COLS if c in e.columns]].reset_index(drop=True)

def run_pipeline(src_folder, out_folder, label, week_monday):
    ws=week_monday.strftime('%Y_%m_%d')
    src=next((pathlib.Path(src_folder)/n for n in [f'IEX_Structured_{ws}.csv',f'IEX_Adjusted_{ws}.csv']
              if (pathlib.Path(src_folder)/n).exists()),None)
    if src is None: print(f"[{label}] No file for week {ws}"); return pd.DataFrame()
    print(f"[{label}] Source: {src.name}")
    sd=build_sorted_df(str(src)); iv=generate_intervals(sd)
    out=pathlib.Path(out_folder)/f'{week_monday.strftime("%Y-%m-%d")}.csv'
    iv.to_csv(out,index=False,encoding='utf-8-sig')
    print(f"[{label}] {len(iv):,} rows saved")
    return iv

print("Helpers loaded.")

def parse_actual_iex_file(fp, week_monday=None):
    raw = pd.read_excel(fp, header=None, dtype=str)
    COL = {'agent':1,'date':2,'ss':3,'es':4,'act':6,'sa':7,'ea':10}
    SKIP = {'Agent Schedules','Date Range:','MU:','Date','nan','','None'}
    rows=[]; cur_agent=cur_date=cur_ss=cur_es=None
    for _, r in raw.iterrows():
        def _v(c): return str(r.get(c,'') if c<len(r) else '').strip().replace('nan','')
        col1=_v(COL['agent']); col2=_v(COL['date']); col6=_v(COL['act'])
        if col1.startswith('Agent:'):
            cur_agent=col1; cur_date=cur_ss=cur_es=None; continue
        if cur_agent is None: continue
        if not col6 or col6 in SKIP or col6=='Scheduled Activity': continue
        if col2 and col2 not in SKIP:
            cur_date=col2; ss=_v(COL['ss']); es=_v(COL['es'])
            if ss: cur_ss=ss
            if es: cur_es=es
        if cur_date:
            rows.append({'Agent':cur_agent,'Date':cur_date,'Start_Shift':cur_ss,
                         'End_Shift':cur_es,'Scheduled Activity':col6,
                         'Start_Action':_v(COL['sa']) or None,'End_Action':_v(COL['ea']) or None,
                         'sheet_name':fp.stem})
    df=pd.DataFrame(rows)
    if df.empty: return df
    df['Date']=pd.to_datetime(df['Date'],errors='coerce',dayfirst=False)
    df['IEX_ID']=df['Agent'].str.extract(r'(\d+)',expand=False).astype('Int64')
    if week_monday is not None:
        wend=week_monday+pd.Timedelta(days=6)
        df=df[(df['Date']>=week_monday)&(df['Date']<=wend)]
    return df.sort_values(['Agent','Date','Start_Action'],na_position='first').reset_index(drop=True)

print("Helpers loaded.")

Helpers loaded.
Helpers loaded.


In [8]:
print('Processing ORIGINAL intervals ...')
iv_original = run_pipeline(PATHS['structured_v1'], PATHS['intervals_original'], 'ORIGINAL', _week)
print(f'  → {len(iv_original):,} rows | cols: {list(iv_original.columns)[:5]}')

print('\nProcessing ADJUSTED intervals ...')
iv_adjusted = run_pipeline(PATHS['after_adjustment'], PATHS['intervals_after_adjustment'], 'ADJUSTED', _week)
print(f'  → {len(iv_adjusted):,} rows | cols: {list(iv_adjusted.columns)[:5]}')

def _load_iv(folder, week):
    p=pathlib.Path(folder)/f'{week.strftime("%Y-%m-%d")}.csv'
    if not p.exists():
        print(f'  [WARN] Not found: {p}')
        return pd.DataFrame()
    df=pd.read_csv(p); df['Date_Converted']=pd.to_datetime(df['Date_Converted'],errors='coerce')
    return df

if iv_original.empty:
    print('Trying to load ORIGINAL from saved CSV...')
    iv_original=_load_iv(PATHS['intervals_original'],_week)
    print(f'  Loaded: {len(iv_original):,} rows')

if iv_adjusted.empty:
    print('Trying to load ADJUSTED from saved CSV...')
    iv_adjusted=_load_iv(PATHS['intervals_after_adjustment'],_week)
    print(f'  Loaded: {len(iv_adjusted):,} rows')

for label, iv, src_folder in [('ORIGINAL',iv_original,PATHS['structured_v1']),
                               ('ADJUSTED',iv_adjusted,PATHS['after_adjustment'])]:
    if iv.empty:
        print(f'\n[DIAG] {label} is empty. Checking source folder...')
        ws=_week.strftime('%Y_%m_%d')
        for pat in [f'IEX_Structured_{ws}.csv', f'IEX_Adjusted_{ws}.csv']:
            p=pathlib.Path(src_folder)/pat
            print(f'  {"✅" if p.exists() else "❌"} {pat}')
        print(f'  All files in folder: {[f.name for f in pathlib.Path(src_folder).glob("*.csv")]}')

print(f'\nFinal: iv_original={len(iv_original):,} | iv_adjusted={len(iv_adjusted):,}')


Processing ORIGINAL intervals ...
[ORIGINAL] Source: IEX_Structured_2026_09_07.csv
[ORIGINAL] 11,432 rows saved
  → 11,432 rows | cols: ['Month', 'Week_Monday', 'Date_Converted', 'Agent Name', 'IEX ID']

Processing ADJUSTED intervals ...
[ADJUSTED] Source: IEX_Adjusted_2026_09_07.csv
[ADJUSTED] 11,231 rows saved
  → 11,231 rows | cols: ['Month', 'Week_Monday', 'Date_Converted', 'Agent Name', 'IEX ID']

Final: iv_original=11,432 | iv_adjusted=11,231


In [9]:
def _read_vnm_ou_sheet(fp):
    from datetime import datetime as _dtt, timedelta
    DAY_NAMES = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    xl = pd.ExcelFile(fp)
    sheet = next((s for s in xl.sheet_names if s.strip().upper()=='VNM - OU'), None)
    if sheet is None:
        sheet = next((s for s in xl.sheet_names if 'VNM' in s.upper()), None)
    if sheet is None:
        print(f'    [WARN] No VNM sheet in {fp.name}'); return pd.DataFrame()
    print(f'    Sheet: "{sheet}"')

    df = pd.read_excel(fp, sheet_name=sheet, header=None, skiprows=5, nrows=48)
    print(f'    Raw shape: {df.shape}')

    # Col layout: A(0)=PST, then 3-col groups per day starting at E(4)
    # E=Mon Req W, F=Mon Prov, G=Mon DIFF | H=Tue Req W, I=Tue Prov, J=Tue DIFF | ...
    def _to_hhmm(v):
        if hasattr(v,'hour'): return f'{v.hour:02d}:{v.minute:02d}'
        s=str(v).strip()
        for fmt in ('%H:%M:%S','%H:%M'):
            try: return _dtt.strptime(s,fmt).strftime('%H:%M')
            except: pass
        return None

    pst_col = df.iloc[:, 0].apply(_to_hhmm)
    valid   = pst_col.notna()
    pst_col = pst_col[valid]

    rows = []
    for idx in pst_col.index:
        t = pst_col[idx]
        try:
            t_end = (_dtt.combine(_dtt.today(), _dtt.strptime(t,'%H:%M').time())
                     + timedelta(minutes=30)).strftime('%H:%M')
        except: continue
        row = {'PST_Interval_Range': f'{t}-{t_end}'}
        for j, day in enumerate(DAY_NAMES):
            req_idx  = 4 + j * 3  # E,H,K,N,Q,T,W
            prov_idx = 5 + j * 3  # F,I,L,O,R,U,X
            try: row[day+'_Req']  = float(df.iloc[idx, req_idx]  or 0) / 2 if df.shape[1]>req_idx  else 0.0
            except: row[day+'_Req']  = 0.0
            try: row[day+'_Prov'] = float(df.iloc[idx, prov_idx] or 0) / 2 if df.shape[1]>prov_idx else 0.0
            except: row[day+'_Prov'] = 0.0
        rows.append(row)

    result = pd.DataFrame(rows)
    if result.empty: return result
    keep = ['PST_Interval_Range']+[d+m for d in DAY_NAMES for m in ('_Req','_Prov')]
    result = result[[c for c in keep if c in result.columns]]
    print(f'    Parsed: {len(result)} intervals | cols: {list(result.columns[:5])}')
    return result

def load_ou_req(ou_folder, week_monday):
    """
    Load and combine LG Chat + NL Chat OU Mail files for the given week.
    Returns DataFrame: PST_Interval_Range + Mon..Sun (Req W / 2).
    """
    import re
    ws_tag = week_monday.strftime('%m_%d_%y')
    all_xlsx = sorted([f for f in pathlib.Path(ou_folder).glob('*.xlsx')
                       if not f.name.startswith('~$')])
    print(f'  OU folder files: {[f.name for f in all_xlsx]}')

    week_files = [f for f in all_xlsx if ws_tag in f.name]
    if not week_files:
        print(f'  [WARN] No files matching {ws_tag}')
        return pd.DataFrame()
    print(f'  Week files: {[f.name for f in week_files]}')

    combined = None
    DAY_NAMES = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    for fp in week_files:
        print(f'  Reading: {fp.name}')
        df = _read_vnm_ou_sheet(fp)
        if df.empty: continue
        if combined is None:
            combined = df.copy()
        else:
            combined = combined.merge(df, on='PST_Interval_Range',
                                      suffixes=('','_r'), how='outer').fillna(0)
            for d in DAY_NAMES:
                for m in ('_Req','_Prov'):
                    c = d+m; cr = c+'_r'
                    if cr in combined.columns:
                        combined[c] = combined[c] + combined.pop(cr)

    if combined is None or combined.empty:
        print('  [WARN] No OU data loaded')
        return pd.DataFrame()

    print(f'  Combined OU Req: {len(combined)} PST intervals')
    print(f'  Sample (first 2):\n{combined.head(2).to_string()}')
    return combined

def build_ou_pivot(ou_df, iv_ref, week_dates, interval_col, all_ivls):
    dl_keys = [d.strftime('%a\n%d/%m') for d in week_dates]
    empty = pd.DataFrame(0.0, index=all_ivls, columns=dl_keys)
    if ou_df.empty or iv_ref.empty or interval_col not in iv_ref.columns:
        return empty
    if interval_col == 'PST_Interval_Range':
        pst_map = {v: v for v in all_ivls}  # identity: PST → PST
    elif 'PST_Interval_Range' in iv_ref.columns:
        pst_map = (iv_ref[[interval_col,'PST_Interval_Range']]
                   .dropna(subset=['PST_Interval_Range'])
                   .drop_duplicates(interval_col)
                   .set_index(interval_col)['PST_Interval_Range']
                   .to_dict())
    else:
        return empty
    DAY_NAMES = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    ou_idx = ou_df.set_index('PST_Interval_Range') if not ou_df.empty and 'PST_Interval_Range' in ou_df.columns else pd.DataFrame()
    req_rows = []; prov_rows = []
    for ivl in all_ivls:
        req_row = {'Interval': ivl}; prov_row = {'Interval': ivl}
        pst_ivl = pst_map.get(ivl)
        for d in week_dates:
            dk = d.strftime('%a\n%d/%m')
            day_name = DAY_NAMES[d.dayofweek]
            req_val = prov_val = 0.0
            if pst_ivl and not ou_idx.empty and pst_ivl in ou_idx.index:
                req_col  = day_name+'_Req'
                prov_col = day_name+'_Prov'
                try: req_val  = float(ou_idx.loc[pst_ivl, req_col]  or 0) if req_col  in ou_idx.columns else 0.0
                except: req_val = 0.0
                try: prov_val = float(ou_idx.loc[pst_ivl, prov_col] or 0) if prov_col in ou_idx.columns else 0.0
                except: prov_val = 0.0
            req_row[dk] = req_val; prov_row[dk] = prov_val
        req_rows.append(req_row); prov_rows.append(prov_row)
    pvt_req  = pd.DataFrame(req_rows).set_index('Interval')
    pvt_prov = pd.DataFrame(prov_rows).set_index('Interval')
    nzr = (pvt_req  != 0).any(axis=1).sum()
    nzp = (pvt_prov != 0).any(axis=1).sum()
    print(f'  OU pivot: req non-zero={nzr} | prov non-zero={nzp}')
    return pvt_req, pvt_prov

def process_actual_intervals(actual_folder, week_monday):
    actual_files=[f for f in
        list(pathlib.Path(actual_folder).glob('**/*.xlsx'))+
        list(pathlib.Path(actual_folder).glob('**/*.csv'))
        if not f.name.startswith('~$')]
    if not actual_files: print('[WARN] IEX_ACTUAL_ADJUSTMENT is empty.'); return pd.DataFrame()
    dfs=[]
    for fp in actual_files:
        df=parse_actual_iex_file(fp,week_monday=week_monday)
        if not df.empty: dfs.append(df)
    if not dfs: return pd.DataFrame()
    combined=pd.concat(dfs,ignore_index=True)
    print(f'  Actual structured: {len(combined):,} rows | {combined["IEX_ID"].nunique()} agents')
    import tempfile
    tmp=pathlib.Path(tempfile.gettempdir())/f'_actual_{week_monday.strftime("%Y_%m_%d")}.csv'
    combined.to_csv(tmp,index=False,encoding='utf-8-sig')
    sd=build_sorted_df(str(tmp)); iv=generate_intervals(sd)
    try: os.remove(tmp)
    except: pass
    out=pathlib.Path(PATHS['intervals_actual'])
    os.makedirs(out,exist_ok=True)
    iv.to_csv(out/f'{week_monday.strftime("%Y-%m-%d")}.csv',index=False,encoding='utf-8-sig')
    print(f'  Actual intervals: {len(iv):,} rows')
    return iv

print('Loading OU Req from OU Mail files (LG + NL Chat combined) ...')
ou_df = load_ou_req(PATHS['ou_mail_folder'], _week)

print('Processing IEX_ACTUAL_ADJUSTMENT intervals...')
iv_actual=process_actual_intervals(PATHS['actual_adjustment'],_week)
if iv_actual.empty:
    _ap=pathlib.Path(PATHS['intervals_actual'])/f'{_week.strftime("%Y-%m-%d")}.csv'
    if _ap.exists():
        iv_actual=pd.read_csv(_ap); iv_actual['Date_Converted']=pd.to_datetime(iv_actual['Date_Converted'],errors='coerce')
        print(f'  Loaded from cache: {len(iv_actual):,} rows')

Loading OU Req from OU Mail files (LG + NL Chat combined) ...
  OU folder files: ['Global OU LG Chat 07_06_26.xlsx', 'Global OU LG Chat 07_13_26.xlsx', 'Global OU LG Chat 07_20_26.xlsx', 'Global OU LG Chat 07_27_26.xlsx', 'Global OU LG Chat 08_03_26.xlsx', 'Global OU LG Chat 08_10_26.xlsx', 'Global OU LG Chat 08_17_26.xlsx', 'Global OU LG Chat 08_24_26.xlsx', 'Global OU LG Chat 08_31_26.xlsx', 'Global OU LG Chat 09_07_26.xlsx', 'Global OU NL Chat 07_06_26.xlsx', 'Global OU NL Chat 07_13_26.xlsx', 'Global OU NL Chat 07_20_26.xlsx', 'Global OU NL Chat 07_27_26.xlsx', 'Global OU NL Chat 08_03_26.xlsx', 'Global OU NL Chat 08_10_26.xlsx']
  Week files: ['Global OU LG Chat 09_07_26.xlsx']
  Reading: Global OU LG Chat 09_07_26.xlsx
    Sheet: "VNM - OU "
    Raw shape: (48, 225)
    Parsed: 48 intervals | cols: ['PST_Interval_Range', 'Mon_Req', 'Mon_Prov', 'Tue_Req', 'Tue_Prov']
  Combined OU Req: 48 PST intervals
  Sample (first 2):
  PST_Interval_Range  Mon_Req  Mon_Prov  Tue_Req  Tue_Prov 

c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_3716\2340387520.py:147: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date']=pd.to_datetime(df['Date'],errors='coerce',dayfirst=False)
c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_3716\2340387520.py:147: UserWarning: Could not infer format, so each element will be parsed individually, falling back 

  Actual structured: 3,696 rows | 108 agents
  Actual intervals: 10,965 rows


In [10]:
def _change_type(first_shift, scheduled_activity=''):
    fs  = str(first_shift or '').strip().upper()
    act = str(scheduled_activity or '').strip().lower()
    mapped = IEX_ACT_TO_PLANNED_TYPE.get(act)
    if mapped: return mapped
    if any(x in fs for x in ['AL','PTO']): return 'AL'
    if any(x in fs for x in ['CO','PAID']):  return 'CO'
    if any(x in fs for x in ['LWP','UNPAID']): return 'LWP'
    if 'TERMINATION' in fs: return 'Termination'
    if any(x in fs for x in ['SICK','SL']):    return 'Sick Leave'
    if any(x in fs for x in ['HOLIDAY','HO']): return 'Holiday'
    if 'NCNS' in fs:    return 'NCNS'
    if 'TRAINING' in fs: return 'Training'
    return 'Swap'

def _lookup_planned(iex_id, date, planned_df):
    if planned_df.empty: return None
    m = planned_df[(planned_df['IEX_ID']==iex_id) &
                   (planned_df['Leave_Date'].dt.normalize()==pd.Timestamp(date).normalize())]
    if m.empty: return None
    return str(m.iloc[0]['Type'])

def build_comparison(orig, adj, planned_df=pd.DataFrame()):
    REQUIRED = {'Work Category','Duration','Date_Converted'}
    def _check(df, label):
        missing = REQUIRED - set(df.columns)
        if df.empty or missing:
            print(f'[WARN] {label} empty or missing {missing}')
            return False
        return True
    if not _check(orig,'iv_original') or not _check(adj,'iv_adjusted'):
        return pd.DataFrame()

    G  = ['Date_Converted','VNT_Interval_Range','PST_Interval_Range']
    G2 = ['Date_Converted','VNT_Interval_Range']
    po = (orig[orig['Work Category']=='Productive'].groupby(G,sort=False)['Duration'].sum().reset_index(name='Productive_Before'))
    pa = (adj[adj['Work Category']=='Productive'].groupby(G,sort=False)['Duration'].sum().reset_index(name='Productive_After'))
    cmp = po.merge(pa,on=G,how='outer').fillna(0)
    cmp['Productive_Delta'] = cmp['Productive_After']-cmp['Productive_Before']
    cmp['Has_Change'] = (cmp['Productive_Delta'].abs()>0.001)

    ao = orig[orig['Work Category']=='Productive'][['Date_Converted','VNT_Interval_Range','IEX ID','Agent Name','First Shift']].rename(columns={'First Shift':'Shift_Before'})
    aa = adj[adj['Work Category']=='Productive'][['Date_Converted','VNT_Interval_Range','IEX ID','Agent Name','First Shift']].rename(columns={'First Shift':'Shift_After'})
    ag = ao.merge(aa,on=['Date_Converted','VNT_Interval_Range','IEX ID','Agent Name'],how='outer')

    adj_info  = (adj.groupby(G2+['IEX ID'],sort=False)
                   .agg(Adj_FS=('First Shift','first'), Adj_Act=('Scheduled Activity','first'))
                   .reset_index())
    orig_info = (orig.groupby(G2+['IEX ID'],sort=False)
                    .agg(Orig_FS=('First Shift','first'), Orig_Act=('Scheduled Activity','first'))
                    .reset_index())

    dropped = ag[ag['Shift_After'].isna()&ag['Shift_Before'].notna()].copy()
    gained  = ag[ag['Shift_Before'].isna()&ag['Shift_After'].notna()].copy()

    if not dropped.empty:
        dropped = dropped.merge(adj_info, on=G2+['IEX ID'], how='left')
        dropped['Type'] = dropped.apply(
            lambda r: _lookup_planned(r['IEX ID'], r['Date_Converted'], planned_df)
                      or _change_type(r.get('Adj_FS',''), r.get('Adj_Act','')), axis=1)

    if not gained.empty:
        gained = gained.merge(orig_info, on=G2+['IEX ID'], how='left')
        gained['Type'] = gained.apply(
            lambda r: _lookup_planned(r['IEX ID'], r['Date_Converted'], planned_df)
                      or _change_type(r.get('Orig_FS',''), r.get('Orig_Act','')), axis=1)

    def _fmt(df, tag):
        sc = 'Shift_Before' if tag=='OUT' else 'Shift_After'
        df = df.copy()
        df['d'] = (df['IEX ID'].astype(str)+' - '
                   +df['Agent Name'].fillna('?')+' ('+df[sc].fillna('-')+')'
                   +' ['+df.get('Type',pd.Series('Swap',index=df.index)).fillna('Swap')+']')
        return df.groupby(G2)['d'].apply('\n'.join).reset_index(name='_d')

    ch_out = _fmt(dropped,'OUT') if not dropped.empty else pd.DataFrame(columns=G2+['_d'])
    ch_in  = _fmt(gained, 'IN')  if not gained.empty  else pd.DataFrame(columns=G2+['_d'])

    cmp = cmp.merge(ch_out.rename(columns={'_d':'Agents_OUT'}),on=G2,how='left') if not ch_out.empty else cmp.assign(Agents_OUT='')
    cmp = cmp.merge(ch_in.rename( columns={'_d':'Agents_IN'}), on=G2,how='left') if not ch_in.empty  else cmp.assign(Agents_IN='')
    cmp['Agents_OUT'] = cmp['Agents_OUT'].fillna('')
    cmp['Agents_IN']  = cmp['Agents_IN'].fillna('')
    cmp['Has_Change'] = cmp['Has_Change']|(cmp['Agents_OUT']!='')|(cmp['Agents_IN']!='')
    return cmp.sort_values(['Date_Converted','VNT_Interval_Range']).reset_index(drop=True)

rta_path   = week_date_to_rta_path(_week, PATHS['rta_folder'])
planned_df = load_planned_sheet(rta_path, f'Schedule_{_week.strftime("%Y_%m_%d")}')

print(f'iv_original : {len(iv_original):,} rows')
print(f'iv_adjusted : {len(iv_adjusted):,} rows')
comparison = build_comparison(iv_original, iv_adjusted, planned_df)
if not comparison.empty:
    print(f'Comparison  : {len(comparison):,} rows | {comparison["Has_Change"].sum()} changed')
else:
    print('[INFO] Comparison empty — check iv_original and iv_adjusted.')


  Planned sheet: 816 entries | Types: {'AL': 500, 'CO': 225, 'LWP': 71, 'SHAL': 8, 'WO': 6, 'HO': 4, 'OFF': 2}
iv_original : 11,432 rows
iv_adjusted : 11,231 rows
Comparison  : 336 rows | 332 changed


In [11]:
def build_interval_pivot(orig, adj, week_dates, interval_col, all_ivls):
    REQUIRED = {'Work Category','Duration','Date_Converted'}
    def _check(df, label):
        missing = REQUIRED - set(df.columns)
        if df.empty or missing:
            print(f'  [WARN] {label} empty or missing {missing} — returning zeros')
            return False
        return True

    empty_pvt = pd.DataFrame(0, index=all_ivls,
                             columns=[d.strftime('%a\n%d/%m') for d in week_dates])

    G=['Date_Converted',interval_col]

    if _check(orig,'iv_original') and interval_col in orig.columns:
        oc=(orig[orig['Work Category']=='Productive']
            .groupby(G,sort=False)['Duration'].sum().reset_index(name='Before'))
        oc['DL']=oc['Date_Converted'].dt.strftime('%a\n%d/%m')
        pvb=oc.pivot_table(index=interval_col,columns='DL',values='Before',
                           aggfunc='sum',fill_value=0).reindex(all_ivls,fill_value=0)
    else:
        pvb=empty_pvt.copy()

    if _check(adj,'iv_adjusted') and interval_col in adj.columns:
        ac=(adj[adj['Work Category']=='Productive']
            .groupby(G,sort=False)['Duration'].sum().reset_index(name='After'))
        ac['DL']=ac['Date_Converted'].dt.strftime('%a\n%d/%m')
        pva=ac.pivot_table(index=interval_col,columns='DL',values='After',
                           aggfunc='sum',fill_value=0).reindex(all_ivls,fill_value=0)
    else:
        pva=empty_pvt.copy()

    return pvb, pva

week_dates=sorted(pd.date_range(_week,periods=7,freq='D').normalize().tolist())

print(f'iv_original: {len(iv_original):,} rows | cols: {list(iv_original.columns)[:6]}')
print(f'iv_adjusted: {len(iv_adjusted):,} rows | cols: {list(iv_adjusted.columns)[:6]}')

if iv_original.empty or iv_adjusted.empty:
    print('[ACTION NEEDED] One or both interval DataFrames are empty.')
    print('  → Re-run Cell 7 (process intervals) before this cell.')
    ws=_week.strftime('%Y_%m_%d')
    for label, folder, pat in [
        ('STRUCTURED_V1 source', PATHS['structured_v1'], f'IEX_Structured_{ws}.csv'),
        ('AFTER_ADJ source',     PATHS['after_adjustment'], f'IEX_Adjusted_{ws}.csv'),
        ('ORIGINAL intervals',   PATHS['intervals_original'], f'{_week.strftime("%Y-%m-%d")}.csv'),
        ('ADJUSTED intervals',   PATHS['intervals_after_adjustment'], f'{_week.strftime("%Y-%m-%d")}.csv'),
    ]:
        p=pathlib.Path(folder)/pat
        print(f'  {"✅" if p.exists() else "❌"} {label}: {pat}')

vnt_b,vnt_a=build_interval_pivot(iv_original,iv_adjusted,week_dates,'VNT_Interval_Range',ALL_VNT_INTERVALS)

ALL_PST_INTERVALS_ACTUAL=sorted(
    set(iv_original['PST_Interval_Range'].dropna() if 'PST_Interval_Range' in iv_original.columns else [])
   |set(iv_adjusted['PST_Interval_Range'].dropna() if 'PST_Interval_Range' in iv_adjusted.columns else []),
    key=lambda x:x.split('-')[0] if x else '')
if not ALL_PST_INTERVALS_ACTUAL:
    ALL_PST_INTERVALS_ACTUAL = ALL_VNT_INTERVALS  # fallback

pst_b,pst_a=build_interval_pivot(iv_original,iv_adjusted,week_dates,'PST_Interval_Range',ALL_PST_INTERVALS_ACTUAL)
print('VNT pivot:',vnt_b.shape,'| PST pivot:',pst_b.shape)


iv_original: 11,432 rows | cols: ['Month', 'Week_Monday', 'Date_Converted', 'Agent Name', 'IEX ID', 'First Shift']
iv_adjusted: 11,231 rows | cols: ['Month', 'Week_Monday', 'Date_Converted', 'Agent Name', 'IEX ID', 'First Shift']
VNT pivot: (48, 7) | PST pivot: (48, 7)


In [12]:
week_dates=sorted(pd.date_range(_week,periods=7,freq='D').normalize().tolist())
vnt_b,vnt_a=build_interval_pivot(iv_original,iv_adjusted,week_dates,'VNT_Interval_Range',ALL_VNT_INTERVALS)
ALL_PST_INTERVALS_ACTUAL=sorted(
    set(iv_original['PST_Interval_Range'].dropna() if 'PST_Interval_Range' in iv_original.columns else [])
   |set(iv_adjusted['PST_Interval_Range'].dropna() if 'PST_Interval_Range' in iv_adjusted.columns else []),
    key=lambda x:x.split('-')[0] if x else '')
if not ALL_PST_INTERVALS_ACTUAL: ALL_PST_INTERVALS_ACTUAL=ALL_VNT_INTERVALS
pst_b,pst_a=build_interval_pivot(iv_original,iv_adjusted,week_dates,'PST_Interval_Range',ALL_PST_INTERVALS_ACTUAL)

iv_ref=iv_original if not iv_original.empty else iv_adjusted
pvou_vnt, pvou_vnt_prov=build_ou_pivot(ou_df,iv_ref,week_dates,'VNT_Interval_Range',ALL_VNT_INTERVALS)
pvou_pst, pvou_pst_prov=build_ou_pivot(ou_df,iv_ref,week_dates,'PST_Interval_Range',ALL_PST_INTERVALS_ACTUAL)

if not iv_actual.empty:
    vnt_act,_=build_interval_pivot(iv_actual,iv_actual,week_dates,'VNT_Interval_Range',ALL_VNT_INTERVALS)
    pst_act,_=build_interval_pivot(iv_actual,iv_actual,week_dates,'PST_Interval_Range',ALL_PST_INTERVALS_ACTUAL)
else:
    dl=[d.strftime('%a\n%d/%m') for d in week_dates]
    vnt_act=pd.DataFrame(0.0,index=ALL_VNT_INTERVALS,columns=dl)
    pst_act=pd.DataFrame(0.0,index=ALL_PST_INTERVALS_ACTUAL,columns=dl)

print('Pivots ready: vnt_b/a, pst_b/a, vnt_act, pst_act, pvou_vnt, pvou_pst')


  OU pivot: req non-zero=48 | prov non-zero=48
  OU pivot: req non-zero=48 | prov non-zero=48
Pivots ready: vnt_b/a, pst_b/a, vnt_act, pst_act, pvou_vnt, pvou_pst


In [13]:
def generate_excel(pvou_vnt,pvou_vnt_prov,pvou_pst,pvou_pst_prov,vnt_b,vnt_a,pst_b,pst_a,vnt_act,pst_act,pst_ivls,comparison,week_dates,week_monday,out_path):
    COL={'tb':'1F3864','db':'2E4057','bh':'2471A3','ah':'196F3D','dh':'6E2B8B',
         'ib':'D6DCE4','ic':'FFF2CC','bef':'BDD7EE','gn':'27AE60','ls':'C0392B',
         'nt':'EBEBEB','wt':'FFFFFF','dp':'1A8C4E','dn':'A93226','ch':'2C3E50','rc':'FFFDE7'}
    def fl(c): return PatternFill('solid',fgColor=c)
    def fn(bold=False,color='000000',sz=10): return Font(bold=bold,color=color,size=sz,name='Arial')
    def al(h='center',v='center',wrap=False): return Alignment(horizontal=h,vertical=v,wrap_text=wrap)
    sd=Side(style='thin',color='C0C0C0')
    def bd(): return Border(left=sd,right=sd,top=sd,bottom=sd)

    def ws_sum(ws, pvou, pvou_prov, pvb, pva, pvact, title, ilbl, ilst, wdts, mul, dp=1):
        n=len(wdts); lc=1+n*6; last=get_column_letter(lc); fmt='0.0'
        ws.merge_cells(f'A1:{last}1')
        c=ws['A1']; c.value=title; c.font=fn(True,COL['wt'],13); c.fill=fl(COL['tb']); c.alignment=al()
        ws.row_dimensions[1].height=24
        ws.cell(2,1).value=ilbl; ws.cell(2,1).font=fn(True,COL['wt'],10)
        ws.cell(2,1).fill=fl(COL['tb']); ws.cell(2,1).alignment=al()
        for di,d in enumerate(wdts):
            col=2+di*6
            ws.merge_cells(f'{get_column_letter(col)}2:{get_column_letter(col+5)}2')
            c=ws.cell(2,col); c.value=d.strftime('%a\n%d/%m')
            c.font=fn(True,COL['wt'],10); c.fill=fl(COL['db']); c.alignment=al(wrap=True)
        ws.row_dimensions[2].height=30
        ws.cell(3,1).value='Interval'; ws.cell(3,1).font=fn(True,COL['wt'],9)
        ws.cell(3,1).fill=fl(COL['tb']); ws.cell(3,1).alignment=al()
        SUB_HDRS=[('OU_Req','7D3C98'),('OU_Prov','A569BD'),('Before','2471A3'),('After','196F3D'),('Actual','BA4A00'),('Delta','6E2B8B')]
        for di in range(n):
            col=2+di*6
            for j,(lbl,clr) in enumerate(SUB_HDRS):
                sh=ws.cell(3,col+j); sh.value=lbl
                sh.fill=fl(clr); sh.font=fn(True,COL['wt'],8); sh.alignment=al()
        ws.row_dimensions[3].height=16; ws.freeze_panes='B4'
        dlk=[d.strftime('%a\n%d/%m') for d in wdts]
        def _g(pv,ivl,dk): return float(pv.loc[ivl,dk]) if (ivl in pv.index and dk in pv.columns) else 0.0
        for ri,ivl in enumerate(ilst,start=4):
            rch=False
            for di,dk in enumerate(dlk):
                col=2+di*6
                bvr=_g(pvb,ivl,dk); avr=_g(pva,ivl,dk); actr=_g(pvact,ivl,dk)
                our=_g(pvou,ivl,dk); oupr=_g(pvou_prov,ivl,dk)
                bv=round(bvr*mul,dp); av=round(avr*mul,dp); actv=round(actr*mul,dp)
                ouv=round(our*mul,dp); oupv=round(oupr*mul,dp); dv=round((actr-bvr)*mul,dp)
                if abs(dv)>0.001: rch=True
                def _wc(r,col2,val,raw,fill_clr=None,text_clr='000000',bold=False):
                    cc=ws.cell(r,col2); cc.value=val if raw>0 else None
                    if fill_clr: cc.fill=fl(fill_clr)
                    cc.font=fn(sz=10,bold=bold,color=text_clr)
                    cc.alignment=al(); cc.border=bd(); cc.number_format=fmt
                _wc(ri,col,   ouv,  our,  'E8DAEF')
                _wc(ri,col+1, oupv, oupr, 'D2B4DE')
                _wc(ri,col+2, bv,   bvr,  COL['bef'])
                ac2=ws.cell(ri,col+3); ac2.value=av if avr>0 else None
                ac2.alignment=al(); ac2.border=bd(); ac2.number_format=fmt
                if avr-bvr>0.001:    ac2.fill=fl(COL['gn']); ac2.font=fn(sz=10,bold=True,color=COL['wt'])
                elif avr-bvr<-0.001: ac2.fill=fl(COL['ls']); ac2.font=fn(sz=10,bold=True,color=COL['wt'])
                else:                ac2.fill=fl(COL['nt']); ac2.font=fn(sz=10,color='888888')
                act2=ws.cell(ri,col+4); act2.value=actv if actr>0 else None
                act2.alignment=al(); act2.border=bd(); act2.number_format=fmt
                act_diff=actr-bvr
                if act_diff>0.001:    act2.fill=fl('A9DFBF'); act2.font=fn(sz=10,bold=True)
                elif act_diff<-0.001: act2.fill=fl('F1948A'); act2.font=fn(sz=10,bold=True)
                else:                 act2.fill=fl(COL['nt']); act2.font=fn(sz=10,color='888888')
                dc=ws.cell(ri,col+5); dc.value=dv if abs(dv)>0.001 else None
                dc.number_format=fmt; dc.alignment=al(); dc.border=bd()
                if dv>0.001:    dc.fill=fl(COL['dp']); dc.font=fn(bold=True,color=COL['wt'],sz=10)
                elif dv<-0.001: dc.fill=fl(COL['dn']); dc.font=fn(bold=True,color=COL['wt'],sz=10)
                else:           dc.fill=fl(COL['nt']); dc.font=fn(color='888888',sz=10)
            ic=ws.cell(ri,1); ic.value=ivl
            ic.fill=fl(COL['ic'] if rch else COL['ib'])
            ic.font=fn(bold=rch,sz=9); ic.alignment=al('left'); ic.border=bd()
            ws.row_dimensions[ri].height=14
        ws.column_dimensions['A'].width=14
        for di in range(n):
            col=2+di*6
            for j,w2 in enumerate([7,7,8,8,8,6]):
                ws.column_dimensions[get_column_letter(col+j)].width=w2

    def ws_det(ws, cmp, week_monday):
        DC = ['Date_Converted','VNT_Interval_Range','PST_Interval_Range',
              'Productive_Before','Productive_After','Productive_Delta',
              'Has_Change','Agents_OUT','Agents_IN']
        DW = [13, 14, 14, 12, 12, 10, 10, 35, 35]
        if cmp.empty or 'Date_Converted' not in cmp.columns:
            ws['A1'] = 'No comparison data — run Cells 8-9 first.'
            return
        avail = [c for c in DC if c in cmp.columns]
        co = cmp[avail].copy()
        co['Date_Converted'] = pd.to_datetime(co['Date_Converted']).dt.strftime('%Y-%m-%d')
        for c2 in ['Productive_Before','Productive_After','Productive_Delta']:
            if c2 in co.columns: co[c2] = co[c2].round(2)
        last = get_column_letter(len(avail))
        ws.merge_cells(f'A1:{last}1')
        t = ws['A1']; t.value = f'Comparison Detail — Week {week_monday.strftime("%d %b %Y")}'
        t.font = fn(True,COL['wt'],12); t.fill = fl(COL['ch']); t.alignment = al()
        ws.row_dimensions[1].height = 22
        hds = list(co.columns)
        for ci, h in enumerate(hds, start=1):
            c2 = ws.cell(2, ci); c2.value = h
            c2.font = fn(True,COL['wt'],10); c2.fill = fl(COL['ch'])
            c2.alignment = al(wrap=True); c2.border = bd()
        ws.row_dimensions[2].height = 18
        ws.freeze_panes = 'A3'
        for ri, row in enumerate(co.itertuples(index=False), start=3):
            row_vals = dict(zip(hds, row))
            is_ch = bool(row_vals.get('Has_Change', False))
            for ci, (h, val) in enumerate(row_vals.items(), start=1):
                c2 = ws.cell(ri, ci); c2.value = val; c2.border = bd()
                c2.font = fn(sz=10)
                left_cols = ('Date_Converted','VNT_Interval_Range','PST_Interval_Range')
                wrap_cols = ('Agents_OUT','Agents_IN')
                if h in wrap_cols:
                    c2.alignment = Alignment(horizontal='left', vertical='top', wrap_text=True)
                elif h in left_cols:
                    c2.alignment = al('left')
                else:
                    c2.alignment = al('center')
                if h == 'Has_Change' and val:
                    c2.fill = fl(COL['rc'])
                if h == 'Productive_Delta' and isinstance(val,(int,float)):
                    if val > 0.001:   c2.fill = fl(COL['dp']); c2.font = fn(color=COL['wt'],bold=True,sz=10)
                    elif val < -0.001: c2.fill = fl(COL['dn']); c2.font = fn(color=COL['wt'],bold=True,sz=10)
            if is_ch:
                for ci2 in range(1, len(hds)+1):
                    cell_obj = ws.cell(ri, ci2)
                    if cell_obj.fill.fgColor.rgb in ('00000000','FFFFFFFF','00FFFFFF'):
                        cell_obj.fill = fl(COL['rc'])
            n_out = len(str(row_vals.get('Agents_OUT','')).split('\n'))
            n_in  = len(str(row_vals.get('Agents_IN', '')).split('\n'))
            ws.row_dimensions[ri].height = max(14, min(14 * max(n_out, n_in, 1), 200))
        col_widths = {h: DW[j] for j, h in enumerate(avail) if j < len(DW)}
        for ci, h in enumerate(hds, start=1):
            ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(h, 12)

    wb=Workbook(); wk=week_monday.strftime('%d %b %Y')
    ws1=wb.active; ws1.title='VNT Productive'
    ws_sum(ws1,pvou_vnt,pvou_vnt_prov,vnt_b,vnt_a,vnt_act,f'Productive (VNT) — Week of {wk}','VNT Interval',ALL_VNT_INTERVALS,week_dates,mul=1,dp=1)
    ws2=wb.create_sheet('VNT Heads')
    ws_sum(ws2,pvou_vnt,pvou_vnt_prov,vnt_b,vnt_a,vnt_act,f'Heads (VNT) — Week of {wk}','VNT Interval',ALL_VNT_INTERVALS,week_dates,mul=2,dp=1)
    ws3=wb.create_sheet('PST Productive')
    ws_sum(ws3,pvou_pst,pvou_pst_prov,pst_b,pst_a,pst_act,f'Productive (PST) — Week of {wk}','PST Interval',pst_ivls,week_dates,mul=1,dp=1)
    ws4=wb.create_sheet('PST Heads')
    ws_sum(ws4,pvou_pst,pvou_pst_prov,pst_b,pst_a,pst_act,f'Heads (PST) — Week of {wk}','PST Interval',pst_ivls,week_dates,mul=2,dp=1)
    ws5=wb.create_sheet('Detail'); ws_det(ws5,comparison,week_monday)
    wb.save(out_path)
    print('Excel saved:',out_path)

xl_path=os.path.join(PATHS['excel_output'],f'comparison_{WEEK_MONDAY}.xlsx')
if iv_original.empty and iv_adjusted.empty:
    print('[SKIP] generate_excel: iv_original and iv_adjusted both empty.')
    print('       Run Cell 7 (process intervals) first, then retry.')
else:
    generate_excel(pvou_vnt,pvou_vnt_prov,pvou_pst,pvou_pst_prov,vnt_b,vnt_a,pst_b,pst_a,vnt_act,pst_act,ALL_PST_INTERVALS_ACTUAL,comparison,week_dates,_week,xl_path)

Excel saved: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_SIMULATION\comparison_2026-09-07.xlsx


In [14]:
CSS=('<style>'
    ':root{--gn:#27AE60;--ls:#C0392B;--dp:#1A8C4E;--dn:#A93226;--hi:#F39C12;--ttl:#1F3864;--day:#2E4057;}'
    '.wrap{font-family:Segoe UI,Arial,sans-serif;font-size:11px;overflow-x:auto;}'
    '.rpt-title{font-size:15px;font-weight:bold;color:var(--ttl);padding:8px 0 4px;border-bottom:3px solid var(--ttl);margin-bottom:8px;}'
    '.leg{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:8px;font-size:11px;align-items:center;}'
    '.li{display:flex;align-items:center;gap:4px;}'
    '.lb{width:16px;height:16px;border-radius:3px;border:1px solid #999;}'
    'table{border-collapse:collapse;white-space:nowrap;}'
    'th{padding:4px 7px;border:1px solid #4a6278;text-align:center;font-size:10px;position:sticky;top:0;z-index:2;}'
    'th.td{background:var(--day);color:#fff;} th.tb{background:#2471A3;color:#fff;font-size:9px;font-weight:normal;}'
    'th.ta{background:#196F3D;color:#fff;font-size:9px;font-weight:normal;}'
    'th.tD{background:#6E2B8B;color:#fff;font-size:9px;font-weight:bold;}'
    'th.ti{background:var(--ttl);color:#fff;text-align:left;left:0;z-index:3;}'
    'td{padding:3px 6px;border:1px solid #dde;text-align:right;}'
    'td.iv{text-align:left;font-size:9px;font-family:monospace;background:#D6DCE4;position:sticky;left:0;z-index:1;min-width:110px;}'
    'td.iv.ch{background:#FFF2CC;border-left:3px solid var(--hi);font-weight:bold;}'
    'td.bef{background:#BDD7EE;color:#1A5276;} td.gn{background:var(--gn);color:#fff;font-weight:bold;}'
    'td.ls{background:var(--ls);color:#fff;font-weight:bold;} td.nt{background:#EBEBEB;color:#888;}'
    'td.zr{color:#ccc;} td.dp{background:var(--dp);color:#fff;font-weight:bold;}'
    'td.dn{background:var(--dn);color:#fff;font-weight:bold;} td.dz{background:#EBEBEB;color:#aaa;}'
    '</style>')

def render_html(pvb,pva,week_dates,week_monday,all_ivls,mul=1,dp=1,sfx='Productive'):
    dlk=[d.strftime('%a\n%d/%m') for d in week_dates]
    dhl=[d.strftime('%a<br>%d/%m') for d in week_dates]
    def fmt(v): return f'{v:.1f}'
    r1='<th class=\'ti\' rowspan=\'2\'>VNT Interval</th>'+''.join(f'<th class=\'td\' colspan=\'3\'>{x}</th>' for x in dhl)
    r2=''.join("<th class='tb'>Before</th><th class='ta'>After</th><th class='tD'>Delta</th>" for _ in dlk)
    rows=''
    for ivl in all_ivls:
        has_ch=False; tds=''
        for dk in dlk:
            bvr=float(pvb.loc[ivl,dk]) if (ivl in pvb.index and dk in pvb.columns) else 0.0
            avr=float(pva.loc[ivl,dk]) if (ivl in pva.index and dk in pva.columns) else 0.0
            bv=round(bvr*mul,dp); av=round(avr*mul,dp); dv=round((avr-bvr)*mul,dp)
            if abs(dv)>0.001: has_ch=True
            bc='bef zr' if bvr==0 else 'bef'
            ac='gn' if dv>0.001 else ('ls' if dv<-0.001 else ('nt zr' if avr==0 else 'nt'))
            dc='dp' if dv>0.001 else ('dn' if dv<-0.001 else 'dz')
            dtx=('+' if dv>0 else '')+fmt(dv) if abs(dv)>0.001 else ''
            tds+=(f"<td class='{bc}'>{fmt(bv) if bvr else ''}</td>"
                  f"<td class='{ac}'>{fmt(av) if avr else ''}</td>"
                  f"<td class='{dc}'>{dtx}</td>")
        ic='iv ch' if has_ch else 'iv'
        rows+=f"<tr><td class='{ic}'>{ivl}</td>{tds}</tr>"
    ttl=f"<div class='rpt-title'>{sfx} (VNT) — Week of {week_monday.strftime('%d %b %Y')}</div>"
    leg=("<div class='leg'>"
         "<div class='li'><div class='lb' style='background:#BDD7EE'></div>Before</div>"
         "<div class='li'><div class='lb' style='background:#27AE60'></div>After Up</div>"
         "<div class='li'><div class='lb' style='background:#C0392B'></div>After Down</div>"
         "<div class='li'><div class='lb' style='background:#EBEBEB;border:1px solid #bbb'></div>No change</div>"
         "<div class='li'><div class='lb' style='background:#6E2B8B'></div>Delta per day</div>"
         "<div class='li'>Row yellow = changed</div></div>")
    thead=f"<thead><tr>{r1}</tr><tr>{r2}</tr></thead>"
    return CSS+f"<div class='wrap'>{ttl}{leg}<table>{thead}<tbody>{rows}</tbody></table></div>"

display(HTML(render_html(vnt_b,vnt_a,week_dates,_week,ALL_VNT_INTERVALS,mul=1,dp=1,sfx='Productive')))

In [15]:

print("Parsing IEX_ACTUAL_ADJUSTMENT ...")
actual_files=[f for f in
    list(pathlib.Path(PATHS['actual_adjustment']).glob('**/*.xlsx'))
  + list(pathlib.Path(PATHS['actual_adjustment']).glob('**/*.csv'))
    if not f.name.startswith('~$')]

if not actual_files:
    print("Folder empty — place raw IEX xlsx file and re-run.")
    actual_structured=pd.DataFrame()
else:
    print(f"Files: {[f.name for f in actual_files]}")
    dfs=[]
    for fp in actual_files:
        df=parse_actual_iex_file(fp, week_monday=_week)
        if not df.empty:
            dfs.append(df)
            print(f"  {fp.name}: {len(df):,} rows | {df['IEX_ID'].nunique()} agents")
            print(f"  Activities: {df['Scheduled Activity'].value_counts().head(5).to_dict()}")
        else:
            print(f"  {fp.name}: EMPTY — check column layout")
    actual_structured=pd.concat(dfs,ignore_index=True) if dfs else pd.DataFrame()
    if not actual_structured.empty:
        print(f"\nTotal actual_structured: {len(actual_structured):,} rows | {actual_structured['IEX_ID'].nunique()} agents")

Parsing IEX_ACTUAL_ADJUSTMENT ...
Files: ['20226_08_31.xlsx', '20226_09_07.xlsx']


c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_3716\2340387520.py:147: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date']=pd.to_datetime(df['Date'],errors='coerce',dayfirst=False)
c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


  20226_08_31.xlsx: EMPTY — check column layout
  20226_09_07.xlsx: 3,696 rows | 108 agents
  Activities: {'Open Time': 2004, 'Break': 1012, 'Lunch': 534, 'Termination': 65, 'PTO': 28}

Total actual_structured: 3,696 rows | 108 agents


C:\Users\huuchinh.nguyen\AppData\Local\Temp\ipykernel_3716\2340387520.py:147: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date']=pd.to_datetime(df['Date'],errors='coerce',dayfirst=False)


In [16]:
def _norm_time(v):
    if pd.isna(v) or str(v).strip().lower() in ('','nan','none','nat'): return ''
    s=str(v).strip()
    for fmt in ('%I:%M %p','%H:%M:%S','%H:%M','%I:%M%p'):
        try:
            t=pd.to_datetime(s,format=fmt); h=t.hour
            return f"{h%12 or 12}:{t.minute:02d} {'AM' if h<12 else 'PM'}"
        except: continue
    return s

def _time_to_minutes(v):
    """Convert time string to minutes from midnight for sorting."""
    if not v: return 9999
    try:
        t=pd.to_datetime(str(v).strip(), format='%I:%M %p')
        return t.hour*60+t.minute
    except: return 9999

def _parse_dt(date, time_str):
    """Combine date + time string → datetime for Datetime column."""
    nt=_norm_time(time_str)
    if not nt: return pd.NaT
    try:
        dt=pd.to_datetime(str(date.date())+' '+nt, format='%Y-%m-%d %I:%M %p')
        if dt.hour < 6:
            dt += pd.Timedelta(days=1)
        return dt
    except: return pd.NaT

def build_audit(planned_df, actual_df):
    """
    Side-by-side positional alignment.
    For each (IEX_ID, Date): sort both planned and actual by start time,
    then pair row-by-row. Status:
      Matched      = same activity + same times
      Time Diff    = same activity, different times
      Activity Diff= different activity
      Planned Only = extra planned row
      Actual Only  = extra actual row
    """
    all_days=(set(zip(planned_df['IEX_ID'].astype(str),planned_df['Date'].astype(str)))
             |set(zip(actual_df['IEX_ID'].astype(str), actual_df['Date'].astype(str))))

    agent_map={}
    for df in [planned_df,actual_df]:
        if 'Agent' in df.columns:
            agent_map.update(df[['IEX_ID','Agent']].dropna(subset=['Agent'])
                             .drop_duplicates('IEX_ID').set_index('IEX_ID')['Agent'].to_dict())

    rows=[]
    for iex_str,date_str in sorted(all_days):
        iex_id=pd.array([iex_str],dtype='Int64')[0]; date=pd.Timestamp(date_str)
        agent=agent_map.get(iex_id,str(iex_id))

        p_day=planned_df[(planned_df['IEX_ID']==iex_id)&(planned_df['Date']==date)].copy()
        a_day=actual_df[(actual_df['IEX_ID']==iex_id)&(actual_df['Date']==date)].copy()

        p_day['_m']=p_day['Start_Action'].apply(_time_to_minutes)
        a_day['_m']=a_day['Start_Action'].apply(_time_to_minutes)
        if not p_day.empty:
            min_m=p_day['_m'][p_day['_m']<9999].min() if len(p_day)>0 else 0
            if min_m<720: p_day['_m']=p_day['_m'].apply(lambda x: x+1440 if x<min_m else x)
        if not a_day.empty:
            min_m=a_day['_m'][a_day['_m']<9999].min() if len(a_day)>0 else 0
            if min_m<720: a_day['_m']=a_day['_m'].apply(lambda x: x+1440 if x<min_m else x)

        p_day=p_day.sort_values('_m',na_position='last').reset_index(drop=True)
        a_day=a_day.sort_values('_m',na_position='last').reset_index(drop=True)

        for idx in range(max(len(p_day),len(a_day))):
            pr=p_day.iloc[idx].to_dict() if idx<len(p_day) else None
            ar=a_day.iloc[idx].to_dict() if idx<len(a_day) else None

            def _s(d,k): return str(d.get(k,'')) if d and d.get(k) not in (None,'nan','') else ''

            pa=_s(pr,'Scheduled Activity'); aa=_s(ar,'Scheduled Activity')
            ps=_norm_time(_s(pr,'Start_Action')); ps_e=_norm_time(_s(pr,'End_Action'))
            as_=_norm_time(_s(ar,'Start_Action')); as_e=_norm_time(_s(ar,'End_Action'))

            if pr is not None and ar is not None:
                if pa==aa and ps==as_ and ps_e==as_e: status='Matched'
                elif pa==aa:                           status='Time Diff'
                else:                                  status='Activity Diff'
            elif pr is not None: status='Planned Only'
            else:                status='Actual Only'

            start_for_dt=ps or as_
            rows.append({
                'IEX_ID'    : iex_id,
                'Agent'     : agent,
                'Date'      : date,
                'Datetime'  : _parse_dt(date, start_for_dt),
                'P_Activity': pa,
                'P_Shift'   : _s(pr,'Start_Shift')+'-'+_s(pr,'End_Shift') if pr else '',
                'P_Start'   : ps,
                'P_End'     : ps_e,
                'A_Activity': aa,
                'A_Shift'   : _s(ar,'Start_Shift')+'-'+_s(ar,'End_Shift') if ar else '',
                'A_Start'   : as_,
                'A_End'     : as_e,
                'Status'    : status,
            })

    df=pd.DataFrame(rows)
    if not df.empty:
        df=df.sort_values(['IEX_ID','Datetime'],na_position='last').reset_index(drop=True)
    return df

ws_str=_week.strftime('%Y_%m_%d')
planned_path=str(next((p for p in pathlib.Path(PATHS['after_adjustment']).glob(f'IEX_Adjusted_{ws_str}.csv')),
                       pathlib.Path('__NOT_FOUND__')))
print(f'Planned: {planned_path}')

if os.path.isfile(planned_path):
    planned=pd.read_csv(planned_path,dtype=str)
    planned['Date']=pd.to_datetime(planned['Date'],errors='coerce')
    planned['IEX_ID']=planned['Agent'].str.extract(r'(\d+)',expand=False).astype('Int64')
    planned=planned[(planned['Date']>=_week)&(planned['Date']<=_week_end)].copy()
    print(f'Planned: {len(planned):,} rows')
else:
    planned=pd.DataFrame(); print('Planned file not found.')

if not actual_structured.empty:
    actual=actual_structured.copy()
    print(f'Actual : {len(actual):,} rows')
else:
    actual=pd.DataFrame(); print('Actual: empty — re-run Cell 10')

if not planned.empty or not actual.empty:
    p2=planned if not planned.empty else pd.DataFrame(columns=['IEX_ID','Agent','Date','Scheduled Activity','Start_Shift','End_Shift','Start_Action','End_Action'])
    a2=actual  if not actual.empty  else pd.DataFrame(columns=['IEX_ID','Agent','Date','Scheduled Activity','Start_Shift','End_Shift','Start_Action','End_Action'])
    audit_df=build_audit(p2,a2)
    n_m  =(audit_df['Status']=='Matched').sum()
    n_td =(audit_df['Status']=='Time Diff').sum()
    n_ad =(audit_df['Status']=='Activity Diff').sum()
    n_po =(audit_df['Status']=='Planned Only').sum()
    n_ao =(audit_df['Status']=='Actual Only').sum()
    pct  =f'{n_m/len(audit_df)*100:.1f}%' if len(audit_df) else '0%'
    print(f'\nResult: {len(audit_df):,} rows')
    print(f'  Matched      : {n_m:,} ({pct})')
    print(f'  Time Diff    : {n_td:,}')
    print(f'  Activity Diff: {n_ad:,}')
    print(f'  Planned Only : {n_po:,}')
    print(f'  Actual Only  : {n_ao:,}')
else:
    audit_df=pd.DataFrame(); print('No data to compare.')


Planned: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\INPUT_SCHEDULE\SCHEDULE_SIMULATION\IEX_AFTER_ADJUSTMENT\IEX_Adjusted_2026_09_07.csv
Planned: 4,114 rows
Actual : 3,696 rows

Result: 4,384 rows
  Matched      : 3,238 (73.9%)
  Time Diff    : 64
  Activity Diff: 124
  Planned Only : 688
  Actual Only  : 270


In [17]:
if audit_df.empty:
    print('No audit data. Run Cell 12 → Cell 13 first.')
else:
    print(f'Total   : {len(audit_df):,}')
    print(f'Matched : {n_m:,} ({pct})')
    print(f'Time Diff    : {n_td:,}')
    print(f'Activity Diff: {n_ad:,}')
    print(f'Planned Only : {n_po:,}')
    print(f'Actual Only  : {n_ao:,}')

    out_csv = os.path.join(PATHS['excel_output'], f'audit_{WEEK_MONDAY}.csv')
    export_df = audit_df.copy()
    if 'Datetime' in export_df.columns:
        export_df['Datetime'] = export_df['Datetime'].dt.strftime('%Y-%m-%d %I:%M %p')
    export_df.to_csv(out_csv, index=False, encoding='utf-8-sig')
    print(f'\nAudit saved: {out_csv}')


Total   : 4,384
Matched : 3,238 (73.9%)
Time Diff    : 64
Activity Diff: 124
Planned Only : 688
Actual Only  : 270

Audit saved: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_SIMULATION\audit_2026-09-07.csv
